# Business Insights

This notebook turns model outputs into retention-oriented business recommendations.

In [ ]:
import sys
from pathlib import Path

import joblib
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT / 'src'))

from utils import MODELS_DIR, PROCESSED_DIR

X_test = pd.read_csv(PROCESSED_DIR / 'X_test.csv')
y_test = pd.read_csv(PROCESSED_DIR / 'y_test.csv').squeeze('columns')
model = joblib.load(MODELS_DIR / 'final_model.pkl')

scores = X_test.copy()
scores['actual_churn'] = y_test.values
scores['churn_probability'] = model.predict_proba(X_test)[:, 1]
scores[['actual_churn', 'churn_probability']].sort_values('churn_probability', ascending=False).head(10)

In [ ]:
segments = pd.qcut(scores['churn_probability'], q=4, labels=['low', 'medium', 'high', 'critical'])
summary = (
    scores.assign(risk_segment=segments)
    .groupby('risk_segment', observed=True)
    .agg(
        customers=('churn_probability', 'size'),
        avg_probability=('churn_probability', 'mean'),
        actual_churn_rate=('actual_churn', 'mean'),
    )
)
summary

## Recommended Actions

- Prioritize customers in the `critical` risk segment for retention campaigns.
- Offer contract migration incentives to month-to-month customers.
- Bundle tech support or online security when `SupportGap` is present.
- Track campaign cost, saved revenue, recall, and false negatives after each campaign.